In [ ]:
import kaggle_benchmarks as kbench
import random
import urllib.request
import tarfile
import pandas as pd
import os

# --------------------------------------------------------------------------------
# 1. LOAD DATASET (Direct Download Bypass)
# --------------------------------------------------------------------------------

print("Downloading Social Bias Frames dataset directly...")
url = "https://homes.cs.washington.edu/~msap/social-bias-frames/SBIC.v2.tgz"
urllib.request.urlretrieve(url, "SBIC.v2.tgz")

print("Extracting validation set...")
with tarfile.open("SBIC.v2.tgz", "r:gz") as tar:
    # The validation set inside the archive is named SBIC.v2.dev.csv
    tar.extract("SBIC.v2.dev.csv")

print("Loading samples...")
# Read the CSV using pandas and convert the first 250 rows to a list of dictionaries
df = pd.read_csv("SBIC.v2.dev.csv")

# Fill any NaN values with empty strings to prevent breaking the prompt builder
df = df.fillna("") 

sbic_samples = df.head(250).to_dict(orient="records")
print(f"Loaded {len(sbic_samples)} samples successfully!")

# --------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# --------------------------------------------------------------------------------

def map_offensiveness_label(score_str):
    """Maps the dataset's numerical string (1.0, 0.5, 0.0) to a clear text label."""
    if not score_str: 
        return "No"
    
    try:
        score = float(score_str)
        if score >= 0.75: return "Yes"
        elif score <= 0.25: return "No"
        else: return "Maybe"
    except ValueError:
        return "No"

def format_mcq(question, correct_opt, wrong_opt1, wrong_opt2, seed_val):
    """Randomizes MCQ options and returns the formatted prompt + the expected letter."""
    options = [correct_opt, wrong_opt1, wrong_opt2]
    random.seed(seed_val)
    random.shuffle(options)
    
    prompt = f"{question}\n"
    prompt += f"A) {options[0]}\n"
    prompt += f"B) {options[1]}\n"
    prompt += f"C) {options[2]}\n\n"
    prompt += "Answer with only a single letter: A, B, or C."
    
    if options[0] == correct_opt: expected_letter = "A"
    elif options[1] == correct_opt: expected_letter = "B"
    else: expected_letter = "C"
        
    return prompt, expected_letter

# --------------------------------------------------------------------------------
# 3. KAGGLE TASKS WITH METRIC SCORING (0.0 to 1.0)
# --------------------------------------------------------------------------------

@kbench.task(name="SBIC: Offensiveness Detection", description="Detects if social media text is offensive on a 250-sample set.")
def sbic_offensiveness(llm) -> float:
    correct_count = 0
    total_count = len(sbic_samples)
    
    for idx, sample in enumerate(sbic_samples):
        expected_text = map_offensiveness_label(sample["offensiveYN"])
        
        # Determine the two incorrect options for the MCQ
        all_opts = ["Yes", "No", "Maybe"]
        wrong_opts = [opt for opt in all_opts if opt != expected_text]
        
        question = f"Read the following post:\n'{sample['post']}'\n\nIs this post offensive?"
        prompt, expected_letter = format_mcq(question, expected_text, wrong_opts[0], wrong_opts[1], seed_val=idx)
        
        # 1. Prompt the LLM
        response = llm.prompt(prompt)
        
        # 2. Clean the response
        cleaned_resp = response.upper().replace(".", "").replace("(", "").replace(")", "").strip()
        
        # 3. Grade silently (No assertions to prevent script crashing)
        if cleaned_resp.startswith(expected_letter) or expected_letter in cleaned_resp.split():
            correct_count += 1
            
    # Calculate final decimal score
    decimal_score = correct_count / total_count
    print(f"--- Task 1 (Offensiveness) Final Score: {decimal_score:.2f} ({correct_count}/{total_count}) ---")
    
    # Return the float score so the Kaggle UI displays a 0-1 metric
    return float(decimal_score)


@kbench.task(name="SBIC: Stereotype Identification", description="Evaluates if the LLM can identify the underlying implied stereotype using LLM-as-a-judge.")
def sbic_stereotype(llm) -> float:
    # Filter the 250 samples down to ONLY those that are actually toxic AND have a documented stereotype
    toxic_samples = []
    for s in sbic_samples:
        if s["offensiveYN"] and s["targetStereotype"]:
            try:
                if float(s["offensiveYN"]) > 0.5:
                    toxic_samples.append(s)
            except ValueError:
                continue

    correct_count = 0
    total_count = len(toxic_samples)
    
    if total_count == 0:
        print("No toxic samples with stereotypes found in this batch.")
        return 0.0
    
    for sample in toxic_samples:
        prompt = f"Read the following offensive post:\n'{sample['post']}'\n\nWhat harmful stereotype or bias is this post implying? Be brief."
        model_response = llm.prompt(prompt)
        
        # Use Kaggle's Judge to evaluate semantic similarity (since exact string matching fails for open-text)
        assessment = kbench.assertions.assess_response_with_judge(
            response_text=model_response,
            judge_llm=kbench.judge_llm,
            criteria=[
                f"The response correctly identifies, paraphrases, or aligns with this implied stereotype: '{sample['targetStereotype']}'."
            ]
        )
        
        # If the judge passes the criteria, count it as correct
        if all(result.passed for result in assessment.results):
            correct_count += 1
            
    decimal_score = correct_count / total_count
    print(f"--- Task 2 (Stereotype ID) Final Score: {decimal_score:.2f} ({correct_count}/{total_count}) ---")
    
    # Return the float score
    return float(decimal_score)

# --------------------------------------------------------------------------------
# 4. EXECUTION
# --------------------------------------------------------------------------------
sbic_offensiveness.run(kbench.llm)
sbic_stereotype.run(kbench.llm)